In [ ]:
from vizdoom import *
import cv2
import numpy as np
import time
import random
from matplotlib import pyplot as plt
import pandas as pd
import re

# Gym imports
import gymnasium as gym
from gymnasium import Env
from gymnasium.spaces import Discrete, Box

In [ ]:
# Stable Baselines3 imports
from stable_baselines3 import PPO
from stable_baselines3.common import env_checker
from stable_baselines3.common.callbacks import EvalCallback
from stable_baselines3.common.vec_env import DummyVecEnv, VecFrameStack
from stable_baselines3.common.monitor import Monitor

In [ ]:
# Ollama imports
import requests
import json

In [ ]:
# Paths
CONFIG_PATH = './github/ViZDoom/scenarios/take_cover.cfg'
CHECKPOINT_DIR = './train/train_take_cover_llm_reward'
LOG_DIR = './logs/log_take_cover_llm_reward'
MODEL_DIR = './models/take_cover_best_llm_reward'

In [ ]:
class PerceptionLayer:
    URGENCY_IMMINENT   =  200
    URGENCY_APPROACHING = 400
    SIDE_THRESHOLD = 30 

    ACTION_NAMES = {0: "move_left", 1: "move_right"}

    def __init__(self):
        self.prev_health  = 100
        self.last_action  = None
        self.step_count   = 0

    def reset(self):
        self.prev_health = 100
        self.last_action = None
        self.step_count  = 0

    def update(self, state):
        self.step_count += 1

        player_y, health, delta_h = self._get_player_state(state)
        projectiles = self._get_projectiles(state, player_y)

        return self._serialize(player_y, health, delta_h, projectiles)
    
    def update_action(self, action):
        self.last_action = action
    
    def _get_player_state(self, state):
        vars = state.game_variables
        health = vars[0]
        player_y = vars[2]

        delta_h = health - self.prev_health
        self.prev_health = health

        return player_y, health, delta_h
    
    def _get_projectiles(self, state, player_y):
        projectiles = []
        for label in state.labels:
            if label.object_name == "DoomImpBall":
                delta_y = label.object_position_y - player_y
                delta_x = label.object_position_x
                projectiles.append({
                "delta_y": delta_y,
                "delta_x": delta_x,
                "side":    self._side(delta_y),
                "urgency": self._urgency(delta_x),
                })
        
        # Returns projectiles sorted by urgency (closest to player first)
        return sorted(projectiles, key=lambda p: abs(p["delta_x"]))
    
    def _side(self, delta_y: float):
        if abs(delta_y) < self.SIDE_THRESHOLD:
            return "center"
        return "left" if delta_y < 0 else "right"

    def _urgency(self, delta_x: float):
        dist = abs(delta_x)
        if dist < self.URGENCY_IMMINENT:
            return "imminent"
        elif dist < self.URGENCY_APPROACHING:
            return "approaching"
        return "distant"
    
    def _serialize(self, player_y, health, delta_h, projectiles):
        lines = []
        lines.append(f"Health: {int(health)}/100" + (f" (lost {abs(int(delta_h))} HP this step)" if delta_h < 0 else ""))
        lines.append(f"Player position: {self._side(player_y)}")
        lines.append(f"Last action: {self.ACTION_NAMES.get(self.last_action, 'none')}")
        lines.append(f"Survived: {self.step_count} steps")

        if not projectiles:
            lines.append("No projectiles visible.")
        else:
            lines.append(f"{len(projectiles)} projectile(s) detected:")
            for i, p in enumerate(projectiles, 1):
                lines.append(f"  {i}. {p['urgency'].upper()} threat on the {p['side']} side")

        return "\n".join(lines)

In [ ]:
class LLMAgent:
    def __init__(self, model_name, backend_url):
        self.backend_url = backend_url
        self.model_name = model_name
        self.parse_failures = 0

        self.payload = {
            "model" : self.model_name,
            "prompt" : "",
            "stream" : False,
        }

    def evaluate(self, state_text, action_taken):
        prompt  = self._build_prompt(state_text, action_taken)
        response = self._call_llm(prompt)
        reward  = self._parse_reward(response)

        return reward
    
    def _build_prompt(self, state_text, action_taken):
        map_action = {
            0: "move_left",
            1: "move_right",
        }
        
        action_taken = map_action.get(action_taken, "unknown")

        return f"""You are the tactical AI controller for an agent in a take cover video game. Your primary objective is to ensure the agent's survival by effectively dodges the fireballs
            incoming from the enemies. Based on the provided GAME STATE description and ACTION TAKEN by the agent, you must decide the reward for the agent behaviuor.
            
            ### GAME STATE:
            {state_text}

            ### ACTION TAKEN:
            {action_taken}
            
            ### SCORING CRITERIA:
            - Moving away from an IMMINENT threat: +1.0
            - Moving toward an IMMINENT threat: -1.0
            - Moving away from an APPROACHING threat: +0.5
            - No projectiles visible, moving: 0.0
            - Losing HP this step: -1.0 (regardless of action)
            - Dying this step: -1.0
            
            ### OUTPUT FORMAT:
            Respond with ONLY a decimal number between -1.0 and 1.0, nothing else."""

    def _call_llm(self, prompt):
        self.payload["prompt"] = prompt
        self.payload["stream"] = False

        try:
            response = requests.post(self.backend_url, json= self.payload)

            # Check response status
            if response.status_code == 200:
                return response.json()["response"]
            else:
                print(f"Ollama returned error: {response.status_code}")
                return ""
        except requests.exceptions.ConnectionError:
            print("Failed to connect to Ollama backend.")
            return ""   
    
    def _parse_reward(self, response):
        try:
            match = re.search(r"(-?\d+(\.\d+)?)", response) # Extract the first number from the response
            if match:
                value = float(match.group())
                return max(-1.0, min(1.0, value))
            self.parse_failures += 1
            return 0.0 # Fallback if no number found
        except ValueError:
            self.parse_failures += 1
            return 0.0 # Fallback 

In [ ]:
# Create Vizdoom OpenAI Gym Environment
class VizDoomGym(Env): 
    def __init__(self, render=False, llm_agent = None, perception_layer = None): 
        # Setup the game 
        super().__init__()
        self.frame_skip = 4
        self.game = DoomGame()
        self.game.load_config(CONFIG_PATH)
        self.agent = llm_agent
        self.perception = perception_layer
        
        # Render frame logic
        self.game.set_window_visible(render)
        
        # Start the game 
        self.game.init()
        
        # Create the action space and observation space
        self.observation_space = Box(low=0, high=255, shape=(100,160,1), dtype=np.uint8) 
        self.action_space = Discrete(2) # Move left, move right
        self._actions = np.eye(2, dtype=np.uint8)

        # Reward shaping parameters
        # Positve reward for staying alive, negative reward for getting hit and death penalty 
        self.game.set_living_reward(1.0)
        self.game.set_hit_taken_penalty(1.0)
        self.game.set_death_penalty(10.0)
        
    # This is how we take a step in the environment
    def step(self, action):
        # Default reward from the PPO algorithm 
        reward = self.game.make_action(self._actions[action].tolist(), self.frame_skip)
        
        state = self.game.get_state()

        # Get the new state of the game and check if it's done
        if state: 
            obs    = self._process_frame(state.screen_buffer)
            health = state.game_variables[0]
            info   = {"health": health}

            # Get LLM reward and add to the base reward
            if self.agent is not None:
                state_text = self.perception.update(state)
                llm_reward = self.agent.evaluate(state_text, action)
                total_reward = reward + llm_reward
            else:
                total_reward = reward
        else: 
            obs = np.zeros(self.observation_space.shape, dtype=np.uint8)
            info = {"health": 0}
            total_reward = reward
        
        terminated = self.game.is_episode_finished()
        truncated = False

        return obs, total_reward, terminated, truncated, info
    
    # Define how to render the game or environment 
    def render(self): 
        pass
    
    # Starting a new game 
    def reset(self, seed=None, options=None): 
        super().reset(seed=seed)
        self.game.new_episode()
        self.perception.reset()
        state = self.game.get_state()

        if state is not None:
            obs = self._process_frame(state.screen_buffer)
        else:
            obs = np.zeros(self.observation_space.shape, dtype=np.uint8)
        
        return obs, {}
    
    # Call to close down the game
    def close(self): 
        self.game.close()

    def _process_frame(self, buffer: np.ndarray):
        hwc  = np.moveaxis(buffer, 0, -1)                           
        gray = cv2.cvtColor(hwc, cv2.COLOR_RGB2GRAY)               
        resized = cv2.resize(gray, (160, 100), interpolation=cv2.INTER_CUBIC)
        clipped = np.clip(resized, 0, 255).astype(np.uint8)         
        return clipped.reshape(100, 160, 1)

#### Testing the LLM Policy Loop

In [ ]:
# Creates the env wrapped with Monitor 
# To allow the sharing of the llm agent instance between the training and evaluation environments, we create it here and pass it to both envs
def make_env(render=False, model_name="llama3.2:3b", backend_url="http://localhost:11434/api/generate"):
    def _init():
        agent = LLMAgent(model_name=model_name, backend_url=backend_url)
        perception = PerceptionLayer()
        env = VizDoomGym(render=render, llm_agent=agent, perception_layer=perception)
        env = Monitor(env)
        return env
    return _init

In [ ]:
# Build a vectorized environment for training
env = DummyVecEnv([make_env(render=False)])
env = VecFrameStack(env, n_stack=4, channels_order='last')

# Environment for a separate evaluation instance
eval_env = DummyVecEnv([make_env(render=False)])
eval_env = VecFrameStack(eval_env, n_stack=4, channels_order='last')

In [ ]:
# Callbacks
# Saving the best model
eval_callback = EvalCallback(
        eval_env,
        best_model_save_path = MODEL_DIR,
        log_path             = LOG_DIR,
        eval_freq            = 10000,            
        n_eval_episodes      = 10,               
        deterministic        = True,
        render               = False,
        verbose              = 1,
)

In [ ]:
model = PPO(
    env = env,
    tensorboard_log=LOG_DIR,
    policy = 'CnnPolicy',
    learning_rate=3e-5,
    n_steps=4096,
    batch_size=64, 
    n_epochs=4,
    clip_range=0.1,
    ent_coef=0.01,
    verbose=1,
)

In [ ]:
model.learn(
    total_timesteps = 1500000, 
    callback = eval_callback,
)

In [ ]:
env.close()